# PyTorch Pretrained Models - Image Classification

## Setup

In [ ]:
# Set up Kaggle API credentials for downloading datasets
import os  # For file and directory operations
import shutil  # For file copying

kaggle_json_path = './kaggle.json'  # Path to your Kaggle API key
kaggle_dir = os.path.expanduser('~/.kaggle')  # Default Kaggle directory
os.makedirs(kaggle_dir, exist_ok=True)  # Create directory if it doesn't exist
shutil.copy(kaggle_json_path, os.path.join(kaggle_dir, 'kaggle.json'))  # Copy API key
os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)  # Set permissions
print('Kaggle API key set up successfully.')

In [ ]:
# Install and import opendatasets, then download the bean leaf lesions dataset from Kaggle
!pip install -q opendatasets
import opendatasets as od
od.download("https://www.kaggle.com/datasets/marquis03/bean-leaf-lesions-classification")

## Imports

In [ ]:
# Third-party libraries
import matplotlib.pyplot as plt
import torch.nn as nn
import pandas as pd
import torch
from PIL import Image
from sklearn.preprocessing import LabelEncoder
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset
from torchsummary import summary
from torchvision import transforms

# Reproducibility
torch.manual_seed(42)

# Set device to GPU if available, else CPU
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"PyTorch version: {torch.__version__}")
print(f"Using device: {device}")


In [ ]:
# Set ROCm/miopen environment variables for AMD GPU stability
import os
os.environ["MIOPEN_DEBUG_DISABLE_FIND_DB"] = "1"
os.environ["HSA_FORCE_FINE_GRAIN_PCIE"] = "1"

## Image data classification with PyTorch

In [ ]:
train_df = pd.read_csv("bean-leaf-lesions-classification/train.csv")
val_df = pd.read_csv("bean-leaf-lesions-classification/val.csv")

train_df["image:FILE"] = "bean-leaf-lesions-classification/" + train_df["image:FILE"]
val_df["image:FILE"] = "bean-leaf-lesions-classification/" + val_df["image:FILE"]

train_df.head()


In [ ]:
print(f"Training set shape: {train_df.shape}")
print(f"Unique categories in training set: {train_df['category'].unique()}")
print(f"Validation set shape: {val_df.shape}")
print(f"Unique categories in validation set: {val_df['category'].unique()}")

In [ ]:
print(train_df['category'].value_counts())

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.ConvertImageDtype(torch.float)  # Ensure pixel values are in [0, 1]
])

In [ ]:
class CustomImageDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.dataframe = dataframe
        self.transform = transform
        self.labels = torch.tensor(dataframe['category']).to(device) # Move labels to the same device as the model
        
    def __len__(self):
        return self.dataframe.shape[0]
    
    def __getitem__(self, idx):
        img_path = self.dataframe.iloc[idx, 0]  # Assuming the first column contains image paths
        label = self.labels[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = (self.transform(image)/255.0).to(device)  # Normalize pixel values to [0, 1]
        
        return image, label

In [ ]:
train_dataset = CustomImageDataset(train_df, transform)
val_dataset = CustomImageDataset(val_df, transform)

In [ ]:
import numpy as np

n_rows = 3
n_cols = 3

fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 12))

for row in range(n_rows):
    for col in range(n_cols):
        image = train_dataset[np.random.randint(0, train_dataset.__len__())][0].cpu()
        axes[row, col].imshow((image*255).squeeze().permute(1,2,0))  # Convert from (C, H, W) to (H, W, C) for displaying
        axes[row, col].axis('off') # Hide axes for better visualization
        
plt.tight_layout()
plt.show()

In [ ]:
LR = 1e-3
BATCH_SIZE = 1  # Reduced for GPU stability
EPOCHS = 15

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
googleenet_model = torch.hub.load('pytorch/vision:v0.10.0', 'googlenet', pretrained=True)

In [ ]:
for param in googleenet_model.parameters():
    param.requires_grad = True  # Unfreeze all layers for fine-tuning

In [ ]:
googleenet_model.fc

In [ ]:
num_classes = len(train_df['category'].unique())
num_classes  # Getting the number of unique categories in the dataset, which is 3 in this case

In [ ]:
googleenet_model.fc = torch.nn.Linear(googleenet_model.fc.in_features, num_classes)
googleenet_model.fc

In [ ]:
googleenet_model.to(device)

In [ ]:
loss_fun = nn.CrossEntropyLoss() # CrossEntropyLoss is suitable for multi-class classification problems, which is the case here with 3 categories
optimizer = Adam(googleenet_model.parameters(), lr=LR)

total_loss_train_plot = []
total_acc_train_plot = []

for epoch in range(EPOCHS):
    total_acc_train = 0
    total_loss_train = 0
    
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = googleenet_model(inputs)
        
        train_loss = loss_fun(outputs, labels)
        total_loss_train += train_loss.item()
        
        train_loss.backward()
        
        train_acc = (outputs.argmax(outputs, axis=1) == labels).sum().item() # Calculate number of correct predictions
        total_acc_train += train_acc
        optimizer.step()
        
    total_loss_train_plot.append(round(total_loss_train / 1000, 4)) # Average loss per batch
    total_acc_train_plot.append(round(total_acc_train / train_loader.dataset.__len__() * 100, 4)) # Average accuracy per sample
    print(f'''Epoch: {epoch+1}/{EPOCHS} | 
          Train Loss: {total_loss_train_plot[-1]} | 
          Train Accuracy: {total_acc_train_plot[-1]}%''')

In [ ]:
with torch.no_grad():
    total_acc_val = 0
    total_loss_val = 0
    
    for input, labels in val_loader:
        outputs = googleenet_model(input)
        
        val_acc = (outputs.argmax(dim=1) == labels).sum().item() # Calculate number of correct predictions
        total_acc_val += val_acc

In [ ]:
print(round(total_acc_test/val_dataset.__len__() * 100, 2))

## Transfer Learning

In [ ]:
googleenet_model = models.googlenet(weights="DEFAULT")

for param in googleenet_model.parameters():
    param.requires_grad = False  # Unfreeze all layers for fine-tuning
    
googleenet_model.fc = torch.nn.Linear(googleenet_model.fc.in_features, num_classes)
googleenet_model.fc.requires_grad_ = True  # Only the final fully connected layer will be trained
googleenet_model.to(device)

In [ ]:
loss_fun = nn.CrossEntropyLoss() # CrossEntropyLoss is suitable for multi-class classification problems, which is the case here with 3 categories
optimizer = Adam(googleenet_model.parameters(), lr=LR)

total_loss_train_plot = []
total_acc_train_plot = []

for epoch in range(EPOCHS):
    total_acc_train = 0
    total_loss_train = 0
    
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = googleenet_model(inputs)
        
        train_loss = loss_fun(outputs, labels)
        total_loss_train += train_loss.item()
        train_loss.backward()
        
        train_acc = (outputs.argmax(dim=1) == labels).sum().item() # Calculate number of correct predictions
        total_acc_train += train_acc
        optimizer.step()
        
    total_loss_train_plot.append(round(total_loss_train / 1000, 4)) # Average loss per batch
    total_acc_train_plot.append(round(total_acc_train / train_loader.dataset.__len__() * 100, 4)) # Average accuracy per sample
    print(f'''Epoch: {epoch+1}/{EPOCHS} | 
          Train Loss: {total_loss_train_plot[-1]} | 
          Train Accuracy: {total_acc_train_plot[-1]}%''')

In [ ]:
with torch.no_grad():
    total_acc_test = 0
    total_loss_test = 0
    
    for input, labels in test_loader:
        outputs = googleenet_model(input)
        
        val_acc = (outputs.argmax(dim=1) == labels).sum().item() # Calculate number of correct predictions
        total_acc_test += val_acc